# CS322 - Gold Trend Prediction with Least-Squares Regression

**Outcome variable (y):** gold closing price (USD/oz, COMEX gold futures), daily, 5 Feb 2016 – 30 Jan 2026 (2,510 trading days)

**Predictor variables (x):** each one is analysed separately

| Analysis | x | Question |
|---|---|---|
| 1 | x₁ = time (years since 5 Feb 2016) | Does gold follow a trend over time? |
| 2 | x₂ = Bitcoin price on the same day (thousand USD) | Does "digital gold" move with real gold? |

**Function forms fitted to each x:**
1. Linear: y = a + b·x
2. Polynomial of degree 2 (quadratic): y = a + b·x + c·x²
3. Exponential: y = a·e^(b·x)

**Data files (same folder as this notebook):**
- `gold_historical_data.csv` (gold futures, Yahoo Finance)
- `bitcoin_historical_data.csv` (BTC/USD daily close, Bitstamp exchange)

Requirements: `pip install numpy pandas matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

gold = pd.read_csv('gold_historical_data.csv', parse_dates=['Date'])[['Date', 'Close']]
btc = pd.read_csv('bitcoin_historical_data.csv', parse_dates=['date'])[['date', 'close']]
btc = btc.rename(columns={'date': 'Date', 'close': 'BTC'})

# Bitcoin trades 7 days a week, gold only on weekdays: keep the gold trading days
df = gold.merge(btc, on='Date', how='left').sort_values('Date').reset_index(drop=True)
print(df.shape, df['Date'].min().date(), '->', df['Date'].max().date(),
      '| missing BTC values:', df['BTC'].isna().sum())
df.describe()

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(df['Date'], df['Close'], color='goldenrod', lw=1.3, label='Gold (USD/oz)')
ax1.set_ylabel('Gold (USD/oz)', color='goldenrod')
ax2 = ax1.twinx()
ax2.plot(df['Date'], df['BTC'], color='tab:purple', lw=1, alpha=0.8, label='Bitcoin (USD)')
ax2.set_ylabel('Bitcoin (USD)', color='tab:purple'); ax2.grid(False)
ax1.set_title('CS322 - Figure 0: Gold and Bitcoin prices 2016-2026')
fig.savefig('CS322_fig0_gold_bitcoin.png', bbox_inches='tight'); plt.show()
print('Correlation between gold and Bitcoin:', round(df['Close'].corr(df['BTC']), 4))

## 1. Least squares from the formula

For a model y ≈ Aβ, least squares chooses β to minimise SSE = Σ(yᵢ − ŷᵢ)².
Setting the derivative to zero gives the **normal equations**:

$$A^T A\,\beta = A^T y$$

Each function form only changes the design matrix **A**:

| form | columns of A | target |
|---|---|---|
| linear | [1, x] | y |
| quadratic | [1, x, x²] | y |
| exponential | [1, x] | ln y  → a = e^β₀, b = β₁ |

The exponential is fitted by taking ln of both sides: ln y = ln a + b·x, which is a straight line. Its R² is computed on the original y (dollars), so all three forms are compared fairly.

**R² vs adjusted R²:** R² = 1 − SSE/SST never decreases when a term (such as x²) is added. Adjusted R² = 1 − (1 − R²)(n − 1)/(n − p − 1) subtracts a penalty for each extra term p, so it only rises if the new term really improves the fit.

In [ ]:
def solve_normal_equations(A, y):
    # Least-squares coefficients from (A^T A) beta = A^T y
    return np.linalg.solve(A.T @ A, A.T @ y)

def fit(x, y, form):
    # Fit one function form; returns (coefficients, predict function, number of predictor terms p)
    if form == 'Linear':
        b = solve_normal_equations(np.column_stack([np.ones_like(x), x]), y)
        return b, lambda t: b[0] + b[1] * t, 1
    if form == 'Quadratic':
        b = solve_normal_equations(np.column_stack([np.ones_like(x), x, x**2]), y)
        return b, lambda t: b[0] + b[1] * t + b[2] * t**2, 2
    if form == 'Exponential':
        b = solve_normal_equations(np.column_stack([np.ones_like(x), x]), np.log(y))
        a, k = np.exp(b[0]), b[1]
        return np.array([a, k]), lambda t: a * np.exp(k * t), 1
    raise ValueError(form)

def equation(form, c):
    if form == 'Linear':      return f'y = {c[0]:.4g} + {c[1]:.4g}x'
    if form == 'Quadratic':   return f'y = {c[0]:.4g} + {c[1]:.4g}x + {c[2]:.4g}x²'
    if form == 'Exponential': return f'y = {c[0]:.4g}·e^({c[1]:.4g}x)'

def scores(y, y_hat, p):
    n = len(y)
    sse = np.sum((y - y_hat) ** 2)
    sst = np.sum((y - y.mean()) ** 2)
    r2 = 1 - sse / sst
    adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return dict(R2=r2, Adj_R2=adj, SSE=sse, RMSE=np.sqrt(sse / n))

FORMS = ['Linear', 'Quadratic', 'Exponential']
COLORS = {'Linear': 'tab:blue', 'Quadratic': 'tab:green', 'Exponential': 'tab:red'}

## 2. Variables

- **y = gold close** in USD/oz
- **x₁ = time in years** since the first data point (0 = 5 Feb 2016, ≈ 9.98 = 30 Jan 2026)
- **x₂ = Bitcoin close in thousands of USD** on the same day (e.g. 84.13 = \$84,130). Scaling to thousands keeps x² small and the normal equations numerically stable; the fit itself is unchanged.

In [ ]:
y = df['Close'].to_numpy(float)
T0 = df['Date'].iloc[0]
x_time = ((df['Date'] - T0).dt.days / 365.25).to_numpy()
x_btc = df['BTC'].to_numpy(float) / 1000

analyses = {
    'x1: time': dict(x=x_time, xlabel='x₁ = years since 5 Feb 2016'),
    'x2: Bitcoin': dict(x=x_btc, xlabel='x₂ = Bitcoin price (thousand USD)'),
}

## 3. Fit all 6 models and compare R², adjusted R², SSE, RMSE

In [ ]:
rows, models = [], {}
for name, d in analyses.items():
    for form in FORMS:
        c, f, p = fit(d['x'], y, form)
        models[(name, form)] = f
        rows.append({'Predictor': name, 'Form': form, 'Equation': equation(form, c),
                     **scores(y, f(d['x']), p)})
results = pd.DataFrame(rows)
results.to_csv('regression_results.csv', index=False)
pd.set_option('display.max_colwidth', 60); pd.set_option('display.width', 160)
print('CS322 - Table 1: Least-squares fits of gold price (n = %d)' % len(y))
results.round({'R2': 4, 'Adj_R2': 4, 'SSE': 0, 'RMSE': 2})

## 4. Plots: data with each fitted function

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
d = analyses['x1: time']
ax.scatter(d['x'], y, s=3, color='grey', alpha=0.5, label='Data')
xs = np.linspace(d['x'].min(), d['x'].max(), 400)
for form in FORMS:
    r2 = results.query('Predictor == "x1: time" and Form == @form')['R2'].item()
    ax.plot(xs, models[('x1: time', form)](xs), color=COLORS[form], lw=2, label=f'{form}  (R² = {r2:.3f})')
ax.set_xlabel(d['xlabel']); ax.set_ylabel('y = gold close (USD/oz)')
ax.set_title('CS322 - Figure 1: Gold price vs time with least-squares fits'); ax.legend()
fig.savefig('CS322_fig1_time.png', bbox_inches='tight'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
d = analyses['x2: Bitcoin']
sc = ax.scatter(d['x'], y, s=4, c=df['Date'].dt.year, cmap='viridis', alpha=0.7)
plt.colorbar(sc, ax=ax, label='year')
xs = np.linspace(d['x'].min(), d['x'].max(), 400)
for form in FORMS:
    r2 = results.query('Predictor == "x2: Bitcoin" and Form == @form')['R2'].item()
    ax.plot(xs, models[('x2: Bitcoin', form)](xs), color=COLORS[form], lw=2, label=f'{form}  (R² = {r2:.3f})')
ax.set_xlabel(d['xlabel']); ax.set_ylabel('y = gold close (USD/oz)')
ax.set_title('CS322 - Figure 2: Gold price vs Bitcoin price with least-squares fits'); ax.legend(loc='upper left')
fig.savefig('CS322_fig2_bitcoin.png', bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5), sharey=True)
for row, name in zip(axes, analyses):
    for ax, form in zip(row, FORMS):
        ax.scatter(df['Date'], y - models[(name, form)](analyses[name]['x']), s=2, color=COLORS[form])
        ax.axhline(0, color='black', lw=0.8); ax.set_title(f'{name} - {form}', fontsize=10)
    row[0].set_ylabel('residual y - ŷ (USD)')
fig.suptitle('CS322 - Figure 3: Residuals of all six models over time')
fig.tight_layout(); fig.savefig('CS322_fig3_residuals.png', bbox_inches='tight'); plt.show()

## 5. Using the models for prediction

**Example A – time models (x₁):** predict the gold price on a future date.

**Example B – Bitcoin models (x₂):** predict the gold price from the Bitcoin price on the same day.

Both are checked against the real gold price on **9 Nov 2026 ≈ \$4,123/oz** (Fortune, 9 Nov 2026). Bitcoin closed at **\$75,266.85** that day (Bitstamp).

In [ ]:
ACTUAL_GOLD = 4123.0
btc_full = pd.read_csv('bitcoin_historical_data.csv', parse_dates=['date']).set_index('date')['close']

rows = []
for date in ['2026-02-02', '2026-06-30', '2026-11-09']:
    xt = (pd.Timestamp(date) - T0).days / 365.25
    xb = btc_full.get(pd.Timestamp(date), np.nan) / 1000
    for name, xv in [('x1: time', xt), ('x2: Bitcoin', xb)]:
        row = {'Date': date, 'Predictor': name, 'x': round(xv, 3)}
        for form in FORMS:
            row[form] = models[(name, form)](xv)
        row['Actual gold'] = ACTUAL_GOLD if date == '2026-11-09' else np.nan
        rows.append(row)
pred = pd.DataFrame(rows)
print('CS322 - Table 2: Example predictions of gold price (USD/oz)')
pred.round(1)

## 6. Accuracy check on unseen data: fit on 2016–2024, test on 2025–Jan 2026

R² on the data used for fitting only measures **fit**. To measure **prediction**, each model is refitted using only data up to 31 Dec 2024, then used to predict 2025 – Jan 2026, which it never saw.

In [ ]:
train = (df['Date'] <= '2024-12-31').to_numpy()
rows, test_preds = [], {}
for name, d in analyses.items():
    for form in FORMS:
        c, f, p = fit(d['x'][train], y[train], form)
        test_preds[(name, form)] = f(d['x'])
        s = scores(y[~train], f(d['x'][~train]), p)
        rows.append({'Predictor': name, 'Form': form, 'Train R2': scores(y[train], f(d['x'][train]), p)['R2'],
                     'Test R2': s['R2'], 'Test RMSE': s['RMSE']})
test = pd.DataFrame(rows)
print('CS322 - Table 3: Fit on 2016-2024, tested on 2025 - Jan 2026')
test.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, name in zip(axes, analyses):
    ax.plot(df['Date'], y, color='black', lw=1, label='Actual gold')
    for form in FORMS:
        ax.plot(df['Date'], test_preds[(name, form)], color=COLORS[form], lw=1.2, alpha=0.9, label=form)
    ax.axvline(pd.Timestamp('2024-12-31'), color='grey', ls=':')
    ax.set_title(f'{name}'); ax.legend(loc='upper left', fontsize=8)
axes[0].set_ylabel('USD/oz')
fig.suptitle('CS322 - Figure 4: Models fitted to 2016-2024 (left of dotted line), predicting 2025-2026')
fig.tight_layout(); fig.savefig('CS322_fig4_test.png', bbox_inches='tight'); plt.show()